# Chapter 88: AI in Zero Trust / Identity Security

**Volume 4 — Security Operations**

**The contractor authenticated with valid credentials.** The firewall approved every packet.
Zero Trust policies matched. The PEP allowed access. But the attacker who stole those
credentials quietly traversed your environment for weeks — because static policy cannot
distinguish a valid credential from a stolen one. **AI can.**

### What You Will Build — 5 Identity Security Engines

| Demo | Technique | What It Detects |
|------|-----------|----------------|
| **1. UEBA Risk Scorer** | Weighted behavioral baseline anomaly | Insider threat / credential abuse |
| **2. Entity Resolution Engine** | Multi-attribute identity linking | Fragmented identity / shadow accounts |
| **3. Lateral Movement Graph Detector** | Access graph edge novelty + velocity scoring | Attacker pivoting through access graph |
| **4. Continuous Session Risk Scorer** | Multi-signal adaptive risk fusion | Risk-based step-up authentication |
| **5. JIT Privilege Manager** | Dynamic privilege granting with audit trail | Standing privilege / privilege creep |

### The Core Insight
> **Zero Trust's weakness is that identity is the control plane anchor — and identity is stealable.
> AI shifts the question from "Is this credential valid?" to "Does this *pattern* of behavior
> match the entity who owns this credential?"
> That question is fundamentally harder to fake.**
>
> *Think PDP as OSPF route calculation (computes allow/deny), PEP as FIB (enforces it).
> AI is the real-time topology change detector that tells OSPF when a route is suspicious.*

In [ ]:
# pip install anthropic
import os, json, re, math, time, random
import statistics
from dataclasses import dataclass, field
from typing import List, Dict, Optional
from collections import defaultdict

try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
except Exception:
    import getpass
    if not os.environ.get('ANTHROPIC_API_KEY'):
        os.environ['ANTHROPIC_API_KEY'] = getpass.getpass('Enter your Anthropic API key: ')

api_key = os.environ.get('ANTHROPIC_API_KEY', '')
USE_API = bool(api_key)

if USE_API:
    from anthropic import Anthropic
    client = Anthropic()
    print("LLM analyst: ACTIVE (Anthropic API)")
else:
    print("LLM analyst: SIMULATION MODE (set ANTHROPIC_API_KEY for real analysis)")

def llm_analyze(prompt: str, max_tokens: int = 200) -> str:
    if USE_API:
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}]
        )
        return resp.content[0].text.strip()
    return "Simulation: " + prompt[:80] + "..."

print("Setup complete.")

## Demo 1: UEBA Risk Scorer

**UEBA** (User and Entity Behavior Analytics) builds a behavioral baseline per user:
when they work, what they access, how much data they touch, how their behavior
compares to their peer group.

This demo scores access events against that baseline using **pure Python weighted anomaly scoring** —
no sklearn, no ML libraries. Five factors, each weighted by empirical importance:

| Factor | Weight | What It Measures |
|--------|--------|------------------|
| **New resource access** | 35% | Has this user ever touched this resource type? |
| **Off-hours access** | 25% | Outside their documented working hours? |
| **Data volume spike** | 20% | Transfer volume vs rolling 30-day average? |
| **Peer group deviation** | 10% | Unusual compared to same-role colleagues? |
| **Access velocity** | 10% | Resources-per-session vs their average? |

The combined score feeds the **PDP risk input** — not a binary allow/deny at login,
but a continuous risk signal that evolves through the session.

*Network analogy: This is your BGP community-based path preference. A route isn't
"wrong" — but communities on it tell your policy engine to prefer or avoid it.
UEBA attaches behavioral communities to every identity request.*

In [ ]:
# ── Demo 1: UEBA Risk Scorer ───────────────────────────────────────────────────

@dataclass
class UserProfile:
    """Behavioral baseline for one user — built from 30-90 days of historical data."""
    user_id: str
    role: str
    normal_hours: set           # hours of day (0-23) they typically work
    normal_resource_types: set  # resource categories they normally access
    avg_data_volume_mb: float   # rolling 30-day average data transfer per session
    avg_resources_per_session: float
    peer_group: str             # role cluster for peer comparison

@dataclass
class AccessEvent:
    """A single resource access event to be scored."""
    user_id: str
    resource_type: str    # e.g. 'network_device', 'hr_database', 'financial_report'
    hour: int             # 0-23
    data_volume_mb: float
    session_resource_count: int   # how many resources accessed so far this session
    peer_deviation: float         # 0.0 = matches peers, 1.0 = completely atypical vs peer group

# ── Behavioral baselines (in production: computed from logs, stored in graph DB) ──
PROFILES = {
    "alice.chen": UserProfile(
        user_id="alice.chen",
        role="Network Engineer",
        normal_hours=set(range(7, 19)),
        normal_resource_types={"network_device", "monitoring_dashboard", "noc_portal"},
        avg_data_volume_mb=45.0,
        avg_resources_per_session=6.0,
        peer_group="network_ops",
    ),
    "bob.sharma": UserProfile(
        user_id="bob.sharma",
        role="Finance Analyst",
        normal_hours=set(range(9, 18)),
        normal_resource_types={"financial_report", "erp_system", "payroll_portal"},
        avg_data_volume_mb=20.0,
        avg_resources_per_session=3.0,
        peer_group="finance",
    ),
    "carol.muller": UserProfile(
        user_id="carol.muller",
        role="IT Administrator",
        normal_hours=set(range(6, 22)),
        normal_resource_types={"domain_controller", "network_device", "server_mgmt", "monitoring_dashboard"},
        avg_data_volume_mb=80.0,
        avg_resources_per_session=10.0,
        peer_group="it_ops",
    ),
}

def ueba_score(event: AccessEvent) -> dict:
    """
    Score an access event against the user's behavioral baseline.
    Returns score (0.0=normal, 1.0=maximum anomaly) and factor breakdown.
    No ML libraries — pure weighted arithmetic.
    """
    profile = PROFILES.get(event.user_id)
    if not profile:
        # Unknown user: treat as maximally suspicious
        return {"score": 0.90, "factors": {"unknown_user": 1.0},
                "decision": "BLOCK + ALERT", "role": "Unknown"}

    factors = {}

    # Factor 1: New resource type (never in baseline) — weight 35%
    factors["new_resource"] = 0.0 if event.resource_type in profile.normal_resource_types else 0.85

    # Factor 2: Off-hours access — weight 25%
    factors["off_hours"] = 0.0 if event.hour in profile.normal_hours else 0.75

    # Factor 3: Data volume spike vs 30-day average — weight 20%
    if profile.avg_data_volume_mb > 0:
        ratio = event.data_volume_mb / profile.avg_data_volume_mb
        factors["volume_spike"] = min(max((ratio - 1.5) / 4.0, 0.0), 1.0)
    else:
        factors["volume_spike"] = 0.0

    # Factor 4: Peer group deviation (pre-computed externally) — weight 10%
    factors["peer_deviation"] = min(event.peer_deviation, 1.0)

    # Factor 5: Access velocity vs per-session average — weight 10%
    avg = profile.avg_resources_per_session
    velocity_ratio = event.session_resource_count / max(avg, 1)
    factors["velocity"] = min(max((velocity_ratio - 2.0) / 3.0, 0.0), 1.0)

    weights = {
        "new_resource":  0.35,
        "off_hours":     0.25,
        "volume_spike":  0.20,
        "peer_deviation":0.10,
        "velocity":      0.10,
    }
    score = sum(factors[k] * weights[k] for k in factors)

    # PDP adaptive decision
    if score < 0.25:
        decision = "ALLOW"
    elif score < 0.55:
        decision = "STEP-UP MFA"
    elif score < 0.75:
        decision = "BLOCK + NOTIFY USER"
    else:
        decision = "BLOCK + ALERT SOC"

    return {"score": round(score, 3), "factors": factors,
            "decision": decision, "role": profile.role}

# ── Test events ────────────────────────────────────────────────────────────────
print("=" * 70)
print("UEBA RISK SCORER — ZERO TRUST PDP INPUT")
print("=" * 70)

test_events = [
    # User, resource_type,          hour, vol_mb, session_rsrc, peer_dev
    ("alice.chen",  "network_device",    10,   40.0,   5,   0.08),  # normal
    ("alice.chen",  "hr_database",        3,  220.0,  18,   0.90),  # lateral move + exfil
    ("bob.sharma",  "financial_report",  11,   18.0,   3,   0.05),  # normal
    ("bob.sharma",  "network_device",    23,   50.0,   8,   0.85),  # completely wrong role
    ("carol.muller","domain_controller", 14,   75.0,   9,   0.10),  # normal admin
    ("unknown.ext", "financial_report",   2,  900.0,  35,   0.99),  # unknown user
]

descriptions = [
    "Normal network access",
    "Alice -> HR DB at 3am + bulk download",
    "Normal finance access",
    "Finance user on network gear at 11pm",
    "Normal admin session",
    "Unknown external user — finance + bulk",
]

print(f"\n{'Description':<38} {'Score':>6} {'Decision':<22} {'Role'}")
print("-" * 82)

high_risk = []
for (user, rtype, hour, vol, sess_cnt, peer_dev), desc in zip(test_events, descriptions):
    ev = AccessEvent(user, rtype, hour, vol, sess_cnt, peer_dev)
    result = ueba_score(ev)
    print(f"{desc:<38} {result['score']:>6.3f} {result['decision']:<22} {result['role']}")
    if result["score"] >= 0.55:
        high_risk.append((ev, result))

# LLM triage for high-risk events
if high_risk:
    print(f"\n[UEBA] {len(high_risk)} high-risk events — requesting LLM policy recommendation...")
    for ev, result in high_risk[:2]:
        analysis = llm_analyze(
            f"Zero Trust PDP decision needed. User: {ev.user_id} ({result['role']})\n"
            f"Accessed: {ev.resource_type} at {ev.hour:02d}:00. UEBA score: {result['score']:.3f}.\n"
            f"Anomaly factors: {result['factors']}\n"
            f"Recommended Zero Trust action and MITRE ATT&CK technique? Under 70 words.",
            max_tokens=100
        )
        print(f"\n  [{ev.user_id}] -> {analysis}")

print("\n[UEBA] All risk scores forwarded to PDP for combined policy decision.")

## Demo 2: Entity Resolution Engine

Before you can analyze identity behavior, you need to know it's the *same entity*
across different systems. In real enterprises, one employee appears as multiple
fragmented identities:

```
Active Directory:  john.smith
Email system:      jsmith@company.com
Contractor system: john_smith_ext  (from a previous engagement)
HR system:         Employee ID 12345
Dev service acct:  svc-jsmith-dev
```

**Entity Resolution** links these into a single canonical **"golden record"** by finding
shared attributes across records. The match score is computed from weighted attribute similarity:

| Attribute | Weight | Rationale |
|-----------|--------|-----------|
| Email | 35% | Near-unique identifier |
| Name similarity | 25% | Catches variants (john.smith / jsmith) |
| IP / device | 20% | Same workstation across systems |
| Department | 10% | Reduces cross-org false positives |
| Phone | 10% | Stable unique identifier |

Records scoring above **0.65** are linked to the same golden record.
This matters for security: attackers exploit fragmented identity — using old
undeprovisioned accounts or slight name variations to evade detection.

In [ ]:
# ── Demo 2: Entity Resolution Engine ──────────────────────────────────────────

@dataclass
class IdentityRecord:
    """An identity record from one source system."""
    record_id: str
    source_system: str    # AD, email, HR, contractor, service_account, etc.
    name: str
    email: str
    department: str
    device_ids: List[str]   # workstation hostnames or MAC addresses seen
    phone: str
    last_active: str        # ISO date

def normalize_name(name: str) -> str:
    """Normalize name variants: john.smith, jsmith, John Smith -> 'john smith'."""
    name = name.lower().replace(".", " ").replace("_", " ").replace("-", " ")
    # Remove common suffixes like _ext, _dev, svc-
    name = re.sub(r'\b(ext|dev|svc|tmp|old|test)\b', '', name)
    return " ".join(name.split())

def name_similarity(a: str, b: str) -> float:
    """
    Compute Jaccard similarity between name token sets.
    'john smith' vs 'j smith' -> partial match.
    """
    tokens_a = set(normalize_name(a).split())
    tokens_b = set(normalize_name(b).split())
    if not tokens_a or not tokens_b:
        return 0.0
    intersection = tokens_a & tokens_b
    union = tokens_a | tokens_b
    # Partial credit: if one is contained in the other (jsmith vs john smith)
    for ta in tokens_a:
        for tb in tokens_b:
            if ta.startswith(tb[0]) and (ta[:3] == tb[:3]):
                intersection.add(ta)
    return len(intersection) / len(union)

def email_similarity(a: str, b: str) -> float:
    """Exact match on local part (before @), or full match."""
    a_local = a.split("@")[0].lower() if "@" in a else a.lower()
    b_local = b.split("@")[0].lower() if "@" in b else b.lower()
    if a_local == b_local:
        return 1.0
    # Partial: one is prefix of the other (jsmith vs john.smith)
    shorter, longer = sorted([a_local, b_local], key=len)
    if longer.startswith(shorter[:4]) and len(shorter) >= 4:
        return 0.6
    return 0.0

def device_overlap(devices_a: list, devices_b: list) -> float:
    """Fraction of shared device IDs."""
    if not devices_a or not devices_b:
        return 0.0
    set_a, set_b = set(d.lower() for d in devices_a), set(d.lower() for d in devices_b)
    return len(set_a & set_b) / max(len(set_a | set_b), 1)

def entity_resolution_score(rec_a: IdentityRecord, rec_b: IdentityRecord) -> dict:
    """
    Compute a pairwise match score between two identity records.
    Returns match score (0.0-1.0) and contributing factors.
    """
    factors = {
        "email":    email_similarity(rec_a.email, rec_b.email) * 0.35,
        "name":     name_similarity(rec_a.name, rec_b.name)   * 0.25,
        "device":   device_overlap(rec_a.device_ids, rec_b.device_ids) * 0.20,
        "dept":     (1.0 if rec_a.department.lower() == rec_b.department.lower() else 0.0) * 0.10,
        "phone":    (1.0 if rec_a.phone and rec_a.phone == rec_b.phone else 0.0) * 0.10,
    }
    score = sum(factors.values())
    return {"score": round(score, 3), "factors": {k: round(v, 3) for k, v in factors.items()}}

def build_golden_records(records: List[IdentityRecord], threshold: float = 0.65) -> list:
    """
    Cluster identity records into golden records using single-linkage agglomeration.
    Records with pairwise score >= threshold are merged into the same cluster.
    """
    clusters = [{r} for r in records]  # start: each record is its own cluster

    merged = True
    while merged:
        merged = False
        new_clusters = []
        used = set()
        for i, cluster_a in enumerate(clusters):
            if i in used:
                continue
            merged_cluster = set(cluster_a)
            for j, cluster_b in enumerate(clusters):
                if j <= i or j in used:
                    continue
                # Check if any pair across clusters exceeds threshold
                should_merge = any(
                    entity_resolution_score(a, b)["score"] >= threshold
                    for a in cluster_a for b in cluster_b
                )
                if should_merge:
                    merged_cluster |= cluster_b
                    used.add(j)
                    merged = True
            new_clusters.append(merged_cluster)
            used.add(i)
        clusters = new_clusters

    return clusters

# ── Sample identity records from different enterprise systems ──────────────────
records = [
    IdentityRecord("AD-001",   "Active Directory",   "John Smith",     "john.smith@acme.com",  "Network",  ["LAPTOP-JS001","WKSTN-042"], "+49-89-001", "2024-12-10"),
    IdentityRecord("EMAIL-01", "Email System",        "J. Smith",       "jsmith@acme.com",      "Network",  ["LAPTOP-JS001"],              "+49-89-001", "2024-12-11"),
    IdentityRecord("HR-2341",  "HR System",           "John Smith",     "john.smith@acme.com",  "Network",  [],                            "+49-89-001", "2024-10-01"),
    IdentityRecord("CTR-099",  "Contractor System",   "john_smith_ext", "j.smith@contractor.io","Network",  ["LAPTOP-JS001"],              "",           "2023-06-15"),
    IdentityRecord("SVC-301",  "Dev Service Account", "svc-jsmith-dev", "svc-dev@acme.com",     "Engineering",[],                          "",           "2024-11-30"),
    # Completely different user (should NOT merge with above)
    IdentityRecord("AD-002",   "Active Directory",   "Maria Weber",    "m.weber@acme.com",     "Finance",  ["LAPTOP-MW002"],              "+49-89-555", "2024-12-12"),
    IdentityRecord("HR-2342",  "HR System",           "Maria Weber",    "m.weber@acme.com",     "Finance",  [],                            "+49-89-555", "2024-09-01"),
]

print("=" * 65)
print("ENTITY RESOLUTION — PAIRWISE MATCH SCORES")
print("=" * 65)
print(f"{'Record A':<20} {'Record B':<25} {'Score':>6} {'Match?'}")
print("-" * 65)

# Show key pairwise scores
pairs_to_show = [
    (records[0], records[1]),  # AD vs Email — same person, diff format
    (records[0], records[3]),  # AD vs Contractor old account
    (records[0], records[4]),  # AD vs Service account
    (records[0], records[5]),  # AD vs completely different user
    (records[5], records[6]),  # Maria AD vs Maria HR
]
threshold = 0.65
for a, b in pairs_to_show:
    result = entity_resolution_score(a, b)
    match = "LINKED" if result["score"] >= threshold else "separate"
    print(f"{a.record_id+' '+a.source_system[:8]:<20} {b.record_id+' '+b.source_system[:12]:<25} {result['score']:>6.3f} {match}")

# Build and display golden records
print("\n" + "=" * 65)
print("GOLDEN RECORDS (clustered identities)")
print("=" * 65)
golden = build_golden_records(records, threshold=threshold)
for i, cluster in enumerate(golden, 1):
    names = [r.name for r in cluster]
    sources = [r.source_system for r in cluster]
    print(f"\nGolden Record #{i}: {names[0]} ({len(cluster)} linked identities)")
    for r in sorted(cluster, key=lambda x: x.source_system):
        print(f"  [{r.source_system}] {r.record_id}: {r.name} / {r.email}")

# LLM assessment of suspicious linkages
multi_source = [g for g in golden if len(g) > 3]
if multi_source:
    cluster = list(multi_source[0])
    sources = [r.source_system for r in cluster]
    analysis = llm_analyze(
        f"Entity resolution found {len(cluster)} linked identities for one person: "
        f"systems={sources}. Some accounts haven't been used since 2023. "
        f"Security risk of undeprovisioned linked accounts? Recommended action? Under 70 words.",
        max_tokens=100
    )
    print(f"\n[LLM Security Assessment] {analysis}")

## Demo 3: Graph-Based Lateral Movement Detector

Lateral movement is graph traversal. An attacker compromises Node A, uses A's
relationships to reach Node B, pivots to Node C — always with legitimate credentials.
Each individual access looks normal. The *graph structure change* reveals the attack.

This engine maintains an **access graph** where:
- Nodes = users and resources
- Edges = observed (user, resource) access pairs from 90-day baseline

When a new edge appears in the live feed, a risk score is computed:

```
base_score    = 0.50  (any new edge is suspicious)
+ velocity    = new_edges_in_30min / 3.0  (capped at +0.40)
+ off_hours   = +0.20 if hour not in 08-19
+ privilege   = +0.15 if target is a privileged resource
final_score   = min(base + velocity + off_hours + privilege, 1.0)
```

*Network analogy: Think IS-IS LSDB. You know the topology — which routers are neighbors.
A new adjacency forming where none existed is a topology change event that warrants
investigation. Lateral movement detector watches the authentication topology for
new adjacencies.*

In [ ]:
# ── Demo 3: Graph-Based Lateral Movement Detector ─────────────────────────────

# Privileged resources that get extra weight in scoring
PRIVILEGED_RESOURCES = {
    "domain_controller", "certificate_authority", "vault_secrets",
    "backup_server", "hr_database", "payroll_system", "financial_db",
    "all_users_share", "admin_share"
}

class AccessGraph:
    """
    Tracks historical (user, resource) access pairs.
    Detects new edges (never-seen access) and scores them for lateral movement.
    """

    def __init__(self):
        self.known_edges: set = set()  # (user, resource) pairs from 90-day history
        self.recent_new_edges: List[dict] = []  # for velocity window
        self.alerts: List[dict] = []

    def load_baseline(self, edges: list):
        """Load 90-day historical (user, resource) pairs."""
        for user, resource in edges:
            self.known_edges.add((user, resource))
        print(f"[AccessGraph] Baseline: {len(self.known_edges)} known edges loaded")

    def observe(self, user: str, resource: str, timestamp: float, hour: int) -> Optional[dict]:
        """
        Process one access event. Returns alert dict if new edge, None if known.
        """
        edge = (user, resource)

        if edge in self.known_edges:
            return None  # normal, known edge

        # New edge: add to known (don't re-alert same pair tonight)
        self.known_edges.add(edge)

        # Velocity: how many NEW edges from this user in last 30 minutes?
        window_start = timestamp - 1800
        recent_from_user = [
            e for e in self.recent_new_edges
            if e["user"] == user and e["ts"] >= window_start
        ]
        self.recent_new_edges.append({"user": user, "resource": resource, "ts": timestamp})

        # Score computation
        base      = 0.50
        velocity  = min(len(recent_from_user) / 3.0, 0.40)
        off_hours = 0.20 if hour not in range(8, 19) else 0.0
        privilege = 0.15 if resource in PRIVILEGED_RESOURCES else 0.0
        score     = min(base + velocity + off_hours + privilege, 1.0)

        alert = {
            "type": "LATERAL_MOVEMENT",
            "user": user,
            "new_resource": resource,
            "hour": hour,
            "velocity_count": len(recent_from_user),
            "privileged": resource in PRIVILEGED_RESOURCES,
            "score": round(score, 3),
            "components": {
                "base": base, "velocity": round(velocity, 3),
                "off_hours": off_hours, "privilege": privilege
            },
        }
        self.alerts.append(alert)
        return alert

# ── Build graph from 90-day baseline ──────────────────────────────────────────
graph = AccessGraph()
graph.load_baseline([
    # alice.chen — network engineer
    ("alice.chen",   "network_device"),
    ("alice.chen",   "monitoring_dashboard"),
    ("alice.chen",   "noc_portal"),
    # bob.sharma — finance
    ("bob.sharma",   "financial_db"),
    ("bob.sharma",   "payroll_system"),
    ("bob.sharma",   "erp_system"),
    # carol.muller — IT admin
    ("carol.muller", "domain_controller"),
    ("carol.muller", "network_device"),
    ("carol.muller", "vault_secrets"),
    ("carol.muller", "server_mgmt"),
])

# ── Simulate tonight's auth events (attacker using alice's stolen creds) ──────
now = time.time()
print("\n" + "=" * 65)
print("ACCESS GRAPH — TONIGHT'S AUTH EVENTS (02:47 - 03:05)")
print("=" * 65)

tonight = [
    # Legit daytime traffic (pre-loaded to history, no alert)
    (now - 50000, "alice.chen",   "network_device",      9),   # normal
    (now - 48000, "alice.chen",   "monitoring_dashboard",10),  # normal
    # 02:47 — attacker starts moving with alice's credentials
    (now - 900,   "alice.chen",   "hr_database",          2),  # NEW + privileged + off-hours
    (now - 720,   "alice.chen",   "payroll_system",        2),  # NEW + privileged + off-hours
    (now - 540,   "alice.chen",   "domain_controller",    3),  # NEW + privileged + off-hours
    (now - 360,   "alice.chen",   "backup_server",         3),  # NEW + privileged + off-hours
    (now - 180,   "alice.chen",   "all_users_share",       3),  # NEW + privileged + off-hours
    # Bob normal daytime (no alert)
    (now - 45000, "bob.sharma",   "financial_db",         10),  # normal
]

for ts, user, resource, hour in tonight:
    alert = graph.observe(user, resource, ts, hour)
    if alert:
        sev = "CRITICAL" if alert["score"] >= 0.75 else "HIGH"
        priv_tag = " [PRIVILEGED]" if alert["privileged"] else ""
        print(f"\n[{sev}] {user} -> {resource} @ {hour:02d}:00{priv_tag}")
        print(f"  Score: {alert['score']:.3f}  "
              f"velocity={alert['components']['velocity']:.3f}  "
              f"off_hours={alert['components']['off_hours']}  "
              f"privilege={alert['components']['privilege']}")
        if alert["score"] >= 0.70 and alert["velocity_count"] >= 2:
            analysis = llm_analyze(
                f"Lateral movement detection: {user} accessed {alert['velocity_count']+1} new "
                f"privileged resources in 30 minutes at 03:00. Latest: {resource}. "
                f"Score {alert['score']:.3f}. Zero Trust action? MITRE technique? Under 60 words.",
                max_tokens=90
            )
            print(f"  LLM: {analysis}")
    else:
        print(f"  [OK] {user} -> {resource} @ {hour:02d}:00 (baseline edge)")

print(f"\n[Graph] Total lateral movement alerts: {len(graph.alerts)}")
print("[Graph] Account alice.chen flagged for emergency session termination — awaiting analyst approval")

## Demo 4: Continuous Session Risk Scorer

Traditional authentication is a gate: prove identity at login, get a token, use it for
8 hours. Zero Trust demands **continuous re-evaluation** — the risk score evolves
throughout the session based on ongoing observations.

This demo implements a **multi-signal risk fusion engine** that combines five
independent signal sources into one continuous session risk score:

| Signal | Weight | Source |
|--------|--------|--------|
| Auth strength | 20% | MFA type: hardware key > TOTP > SMS > password-only |
| Device posture | 25% | MDM compliance, patch level, EDR running, disk encryption |
| Network context | 20% | Known location, VPN, IP reputation, TOR/proxy detection |
| Behavioral score | 25% | Session UEBA deviation from this user's baseline |
| Threat intel | 10% | Source IP in breach DB, credential exposure feeds |

The PDP decision adapts to the live risk level — not just at login, but every
time a new resource is accessed or a signal changes.

In [ ]:
# ── Demo 4: Continuous Session Risk Scorer ─────────────────────────────────────

@dataclass
class SessionSignals:
    """All signals available for one session risk evaluation."""
    # Auth signal (0.0 = weakest, 1.0 = strongest)
    auth_type: str           # 'hardware_key', 'totp_mfa', 'sms_mfa', 'password_only'
    auth_age_minutes: int    # how old is the authentication? older = riskier

    # Device posture
    mdm_compliant: bool
    os_patch_current: bool
    edr_running: bool
    disk_encrypted: bool

    # Network context
    known_location: bool     # matches user's registered locations
    on_vpn: bool
    ip_reputation: str       # 'clean', 'proxy', 'tor', 'known_bad'

    # Behavioral score from UEBA (0.0 = matches baseline, 1.0 = maximum deviation)
    behavioral_anomaly: float

    # Threat intelligence
    ip_in_breach_db: bool
    credential_exposed: bool

AUTH_STRENGTH = {
    "hardware_key":  0.0,   # strongest auth = zero risk
    "totp_mfa":      0.15,
    "sms_mfa":       0.35,  # SMS is phishable
    "password_only": 0.80,  # no MFA = high auth risk
}

IP_REPUTATION_RISK = {
    "clean":     0.0,
    "proxy":     0.4,
    "tor":       0.85,
    "known_bad": 1.0,
}

def compute_session_risk(signals: SessionSignals, user_id: str) -> dict:
    """
    Fuse five signal categories into a single continuous session risk score.
    Returns score (0.0 = safe, 1.0 = maximum risk) + per-signal breakdown.
    """
    # 1. Auth strength signal (0.0-1.0)
    auth_base = AUTH_STRENGTH.get(signals.auth_type, 0.80)
    auth_age_penalty = min(signals.auth_age_minutes / 480.0, 0.30)  # 8hr max penalty
    auth_signal = min(auth_base + auth_age_penalty, 1.0)

    # 2. Device posture signal (0.0-1.0 risk)
    posture_checks = [
        signals.mdm_compliant, signals.os_patch_current,
        signals.edr_running, signals.disk_encrypted
    ]
    device_signal = 1.0 - (sum(posture_checks) / len(posture_checks))

    # 3. Network context signal
    net_risk = IP_REPUTATION_RISK.get(signals.ip_reputation, 0.0)
    if not signals.known_location:
        net_risk = min(net_risk + 0.35, 1.0)
    if signals.on_vpn:
        net_risk = max(net_risk - 0.10, 0.0)  # VPN slightly reduces risk
    net_signal = net_risk

    # 4. Behavioral signal (direct from UEBA)
    behavioral_signal = signals.behavioral_anomaly

    # 5. Threat intelligence signal
    ti_score = 0.0
    if signals.ip_in_breach_db:     ti_score += 0.50
    if signals.credential_exposed:  ti_score += 0.50
    ti_signal = min(ti_score, 1.0)

    # Weighted fusion
    weights = {
        "auth":        0.20,
        "device":      0.25,
        "network":     0.20,
        "behavioral":  0.25,
        "threat_intel":0.10,
    }
    signals_dict = {
        "auth":         auth_signal,
        "device":       device_signal,
        "network":      net_signal,
        "behavioral":   behavioral_signal,
        "threat_intel": ti_signal,
    }
    combined = sum(signals_dict[k] * weights[k] for k in signals_dict)

    # PDP adaptive response
    if combined < 0.20:
        action = "ALLOW — low risk, standard session"
    elif combined < 0.40:
        action = "ALLOW — elevated monitoring, shorter token TTL (15 min)"
    elif combined < 0.60:
        action = "STEP-UP MFA required before continuing"
    elif combined < 0.80:
        action = "BLOCK session — re-authenticate with hardware key"
    else:
        action = "TERMINATE session — lock account — alert SOC immediately"

    return {
        "user": user_id,
        "risk_score": round(combined, 3),
        "signals": {k: round(v, 3) for k, v in signals_dict.items()},
        "pdp_action": action,
    }

# ── Test sessions ──────────────────────────────────────────────────────────────
print("=" * 70)
print("CONTINUOUS SESSION RISK SCORER — ZERO TRUST PDP DECISIONS")
print("=" * 70)

sessions = [
    ("alice.chen", SessionSignals(
        auth_type="totp_mfa", auth_age_minutes=20,
        mdm_compliant=True, os_patch_current=True, edr_running=True, disk_encrypted=True,
        known_location=True, on_vpn=False, ip_reputation="clean",
        behavioral_anomaly=0.05,
        ip_in_breach_db=False, credential_exposed=False
    ), "Normal office session"),

    ("alice.chen", SessionSignals(
        auth_type="password_only", auth_age_minutes=420,
        mdm_compliant=False, os_patch_current=False, edr_running=False, disk_encrypted=False,
        known_location=False, on_vpn=False, ip_reputation="tor",
        behavioral_anomaly=0.88,
        ip_in_breach_db=True, credential_exposed=True
    ), "Stolen creds via TOR — full compromise"),

    ("bob.sharma", SessionSignals(
        auth_type="sms_mfa", auth_age_minutes=90,
        mdm_compliant=True, os_patch_current=True, edr_running=True, disk_encrypted=False,
        known_location=False, on_vpn=True, ip_reputation="proxy",
        behavioral_anomaly=0.35,
        ip_in_breach_db=False, credential_exposed=False
    ), "Bob — remote hotel, unencrypted laptop"),

    ("carol.muller", SessionSignals(
        auth_type="hardware_key", auth_age_minutes=10,
        mdm_compliant=True, os_patch_current=True, edr_running=True, disk_encrypted=True,
        known_location=True, on_vpn=False, ip_reputation="clean",
        behavioral_anomaly=0.08,
        ip_in_breach_db=False, credential_exposed=False
    ), "IT admin — hardware key + managed device"),
]

for user, sigs, desc in sessions:
    result = compute_session_risk(sigs, user)
    print(f"\n{desc}")
    print(f"  User: {user}  |  Risk Score: {result['risk_score']:.3f}")
    sigs_str = "  ".join(f"{k}={v:.2f}" for k, v in result['signals'].items())
    print(f"  Signals: {sigs_str}")
    print(f"  PDP Action: {result['pdp_action']}")

# LLM analysis for the high-risk case
high_risk_user, high_risk_sigs, _ = sessions[1]
high_risk_result = compute_session_risk(high_risk_sigs, high_risk_user)
analysis = llm_analyze(
    f"Zero Trust continuous risk: {high_risk_user} session scored {high_risk_result['risk_score']:.3f}. "
    f"Signals: {high_risk_result['signals']}. Credentials exposed in breach DB, TOR exit node, "
    f"no MFA, unmanaged device, severe behavioral anomaly. Incident response steps? Under 80 words.",
    max_tokens=120
)
print(f"\n[LLM Incident Assessment] {analysis}")

## Demo 5: JIT Privilege Manager

**Standing privilege** is the enemy of Zero Trust. Service accounts with permanent
broad permissions, engineers who accumulated rights across three years of projects,
contractor accounts that were never deprovisioned — all are exploitable with a
single credential theft.

**Just-In-Time (JIT) Privilege** eliminates standing access:
- Privileges are granted dynamically for one specific session and task
- A time-limited token is issued — auto-revoked when it expires
- Every grant is logged with justification and approver
- The ML component checks: does this request match historical patterns for this role?

This demo implements a JIT manager with:
1. **Request scoring**: does this privilege request make sense for this user's role + task?
2. **Token issuance**: cryptographically unique, time-limited, scope-constrained
3. **Audit trail**: immutable log of every grant, use, and revocation
4. **LLM approval assistant**: for unusual requests outside normal patterns

In [ ]:
# ── Demo 5: JIT Privilege Manager ─────────────────────────────────────────────
import hashlib

@dataclass
class JITToken:
    """A time-limited, scope-constrained privilege token."""
    token_id: str
    user_id: str
    privilege_scope: List[str]  # specific permissions granted
    target_resource: str
    issued_at: float            # Unix timestamp
    expires_at: float           # Unix timestamp
    justification: str
    approver: str               # 'auto' for low-risk, 'manager_required' for high-risk
    revoked: bool = False
    use_log: List[str] = field(default_factory=list)

    def is_valid(self) -> bool:
        return not self.revoked and time.time() < self.expires_at

    def ttl_remaining(self) -> float:
        return max(self.expires_at - time.time(), 0.0)

# Role-based privilege templates — what each role CAN request
ROLE_PRIVILEGE_TEMPLATES = {
    "Network Engineer": {
        "allowed_scopes": ["read_device_config", "write_device_config",
                           "restart_interface", "view_routing_table"],
        "max_duration_hours": 4,
        "auto_approve_scopes": ["read_device_config", "view_routing_table"],
        "manager_required_scopes": ["write_device_config", "restart_interface"],
    },
    "Finance Analyst": {
        "allowed_scopes": ["read_financial_report", "write_financial_report",
                           "read_payroll"],
        "max_duration_hours": 8,
        "auto_approve_scopes": ["read_financial_report"],
        "manager_required_scopes": ["read_payroll", "write_financial_report"],
    },
    "IT Administrator": {
        "allowed_scopes": ["domain_admin", "read_all_configs", "write_all_configs",
                           "create_user", "reset_password", "access_vault"],
        "max_duration_hours": 2,
        "auto_approve_scopes": ["read_all_configs", "reset_password"],
        "manager_required_scopes": ["domain_admin", "access_vault", "create_user"],
    },
}

class JITPrivilegeManager:
    """Dynamic JIT privilege granting with audit trail and pattern-based risk scoring."""

    def __init__(self):
        self.active_tokens: Dict[str, JITToken] = {}
        self.audit_log: List[dict] = []
        self._token_counter = 0

    def _generate_token_id(self, user_id: str) -> str:
        self._token_counter += 1
        seed = f"{user_id}-{time.time()}-{self._token_counter}"
        return "JIT-" + hashlib.sha256(seed.encode()).hexdigest()[:12].upper()

    def request_privilege(
        self, user_id: str, role: str,
        requested_scopes: List[str], target_resource: str,
        justification: str, requested_duration_hours: float = 1.0,
        session_risk_score: float = 0.0
    ) -> dict:
        """
        Evaluate and (conditionally) grant a JIT privilege request.

        Returns: grant decision dict with token if approved.
        """
        template = ROLE_PRIVILEGE_TEMPLATES.get(role, {})
        allowed = set(template.get("allowed_scopes", []))
        auto_ok = set(template.get("auto_approve_scopes", []))
        mgr_req = set(template.get("manager_required_scopes", []))
        max_hours = template.get("max_duration_hours", 1)

        # Scope check: are all requested scopes allowed for this role?
        requested_set = set(requested_scopes)
        out_of_role = requested_set - allowed
        if out_of_role:
            self._log_event(user_id, "DENIED", target_resource, requested_scopes,
                            f"Out-of-role scopes: {out_of_role}")
            return {"status": "DENIED", "reason": f"Scopes not allowed for {role}: {out_of_role}"}

        # Duration check
        actual_duration = min(requested_duration_hours, max_hours)

        # Determine approval pathway
        needs_manager = bool(requested_set & mgr_req)

        # High session risk forces manager approval regardless of scope
        if session_risk_score >= 0.55:
            needs_manager = True

        # Grant or pend
        if needs_manager:
            self._log_event(user_id, "PENDING_APPROVAL", target_resource,
                            requested_scopes, justification)
            return {
                "status": "PENDING_MANAGER_APPROVAL",
                "reason": "High-risk scopes or elevated session risk — manager approval required",
                "scopes": requested_scopes,
                "target": target_resource,
            }

        # Auto-approve: issue time-limited token
        token_id = self._generate_token_id(user_id)
        issued_at = time.time()
        token = JITToken(
            token_id=token_id,
            user_id=user_id,
            privilege_scope=requested_scopes,
            target_resource=target_resource,
            issued_at=issued_at,
            expires_at=issued_at + actual_duration * 3600,
            justification=justification,
            approver="auto",
        )
        self.active_tokens[token_id] = token
        self._log_event(user_id, "GRANTED", target_resource, requested_scopes, justification,
                        token_id=token_id, duration_h=actual_duration)
        return {
            "status": "GRANTED",
            "token_id": token_id,
            "scopes": requested_scopes,
            "target": target_resource,
            "valid_for_hours": actual_duration,
            "expires_at": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime(token.expires_at)),
        }

    def revoke(self, token_id: str, reason: str = "manual"):
        if token_id in self.active_tokens:
            self.active_tokens[token_id].revoked = True
            self._log_event(
                self.active_tokens[token_id].user_id, "REVOKED",
                self.active_tokens[token_id].target_resource,
                self.active_tokens[token_id].privilege_scope, reason,
                token_id=token_id
            )

    def _log_event(self, user_id, action, resource, scopes, reason,
                   token_id=None, duration_h=None):
        self.audit_log.append({
            "timestamp": time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime()),
            "user_id": user_id, "action": action, "resource": resource,
            "scopes": scopes, "reason": reason,
            "token_id": token_id, "duration_hours": duration_h,
        })

# ── Test JIT privilege requests ────────────────────────────────────────────────
jit = JITPrivilegeManager()

print("=" * 68)
print("JIT PRIVILEGE MANAGER — REQUEST EVALUATION")
print("=" * 68)

requests = [
    # (user, role, scopes, resource, justification, duration_h, session_risk)
    ("alice.chen",   "Network Engineer", ["read_device_config"],
     "core-router-01", "Troubleshoot OSPF adjacency flap",            1.0, 0.05),

    ("alice.chen",   "Network Engineer", ["write_device_config"],
     "core-router-01", "Apply emergency BGP prefix filter",           0.5, 0.12),

    ("bob.sharma",   "Finance Analyst",  ["read_payroll"],
     "payroll_system",  "Q4 audit report preparation",                2.0, 0.18),

    ("carol.muller", "IT Administrator", ["domain_admin"],
     "domain_controller","Emergency account unlock for CEO",          0.5, 0.08),

    # Suspicious: network engineer requesting out-of-role scope
    ("alice.chen",   "Network Engineer", ["read_payroll", "read_financial_report"],
     "payroll_system", "Checking budget allocation",                  1.0, 0.75),
]

for user, role, scopes, resource, justification, dur, risk in requests:
    result = jit.request_privilege(user, role, scopes, resource, justification, dur, risk)
    print(f"\nRequest: {user} ({role})")
    print(f"  Scopes: {scopes}  Target: {resource}")
    print(f"  Session risk: {risk:.2f}  |  Status: {result['status']}")
    if result["status"] == "GRANTED":
        print(f"  Token: {result['token_id']}  Valid: {result['valid_for_hours']}h  Expires: {result['expires_at']}")
    elif result["status"] == "DENIED":
        print(f"  Reason: {result['reason']}")
        # LLM explains the out-of-role request
        analysis = llm_analyze(
            f"JIT denied: {user} ({role}) requested {scopes} on {resource}. "
            f"This is completely out-of-role for a network engineer. "
            f"Possible insider threat or compromised account? Security recommendation? Under 60 words.",
            max_tokens=90
        )
        print(f"  LLM: {analysis}")
    else:
        print(f"  Reason: {result.get('reason', '')}")

# Audit trail summary
print("\n" + "=" * 68)
print("JIT AUDIT TRAIL")
print("=" * 68)
for entry in jit.audit_log:
    tok = f" [{entry['token_id']}]" if entry["token_id"] else ""
    dur = f" {entry['duration_hours']}h" if entry["duration_hours"] else ""
    print(f"  {entry['timestamp']}  {entry['action']:<22} {entry['user_id']:<15}{tok}{dur}")
    print(f"    Resource: {entry['resource']}  Scopes: {entry['scopes']}")

## Summary: What You Built

You now have a working **AI-Enhanced Zero Trust Identity Security** system with 5 engines:

| Engine | Technique | Key Metric | Threshold |
|--------|-----------|------------|----------|
| **UEBA Risk Scorer** | 5-factor weighted behavioral baseline | Combined anomaly score | > 0.55 = block |
| **Entity Resolution** | Multi-attribute Jaccard similarity | Match score | > 0.65 = same identity |
| **Lateral Movement Graph** | New edge + velocity + privilege weighting | Graph edge score | > 0.70 = critical |
| **Session Risk Scorer** | 5-signal weighted fusion (auth+device+net+UEBA+TI) | Continuous risk score | > 0.60 = step-up |
| **JIT Privilege Manager** | Role-scope matching + token issuance + audit | Role compliance | Out-of-role = deny |

### Why Static Zero Trust Isn't Enough

- **Static policy**: verifies credentials — can't detect stolen credentials used normally
- **AI-enhanced Zero Trust**: verifies *behavioral patterns* — detects credential misuse even when the credential is valid

### The Non-Negotiable Rule

> **All automated blocking decisions require human analyst approval.**
> AI computes the risk score. Humans decide on account termination, SOC escalation,
> or customer impact actions. The ML is the analyst's co-pilot, not the pilot.

### Production Upgrade Path

```
In-memory profiles       ->  Graph DB (Neo4j / Amazon Neptune) with 90-day rolling windows
Static JA3 feed          ->  Real-time threat intel (VirusTotal, Shodan, MISP)
Python dict audit log    ->  Immutable SIEM integration (Splunk / Elastic SIEM)
Hardcoded thresholds     ->  Per-org calibration from 30-day false positive tuning
Simulated LLM triage     ->  claude-sonnet-4-5-20250514 for complex session analysis
```

**Next: Chapter 89 — AI for Cloud & Endpoint Security** — extending the same behavioral
baseline principles to cloud configuration drift, container security, and endpoint
process anomaly detection.

In [ ]:
# ── Chapter 88: Production Deployment Checklist ────────────────────────────────
print("CHAPTER 88: ZERO TRUST + AI — PRODUCTION CHECKLIST")
print("=" * 62)

checklist = [
    ("Identity Data Sources",   "Active Directory / Okta / Azure AD logs -> UEBA engine"),
    ("Identity Data Sources",   "RADIUS / TACACS+ auth logs -> Access graph builder"),
    ("Identity Data Sources",   "HR system + contractor system -> Entity resolution pipeline"),
    ("Identity Data Sources",   "MDM API (Jamf/Intune) -> Device posture signals"),
    ("UEBA Tuning",             "Baseline period: 30 days minimum, 90 days recommended"),
    ("UEBA Tuning",             "Peer group modeling: cluster by role + department"),
    ("UEBA Tuning",             "Cold start: shadow mode only (no blocking) for 30 days"),
    ("UEBA Tuning",             "Factor weights: tune per org risk appetite + FP rate"),
    ("Entity Resolution",       "Threshold 0.65: lower = more merges (more FP), higher = fewer"),
    ("Entity Resolution",       "Deprovisioning sweep: flag stale golden record accounts"),
    ("JIT Privilege",           "Integrate with ServiceNow / Jira for approval workflow"),
    ("JIT Privilege",           "Token TTL: 15-60 min for privileged, 4-8h for read-only"),
    ("JIT Privilege",           "Audit log: write-once, SIEM-integrated, 12-month retention"),
    ("LLM Integration",         "Model: claude-haiku-4-5-20251001 for triage (fast + cheap)"),
    ("LLM Integration",         "Model: claude-sonnet-4-5-20250514 for complex IR analysis"),
    ("Guardrails",              "HUMAN APPROVAL required for: account lock, session kill"),
    ("Guardrails",              "AI recommends — analyst decides — no automated termination"),
    ("Guardrails",              "False positive loop: analyst feedback -> model retraining"),
]

current = ""
for category, item in checklist:
    if category != current:
        print(f"\n[{category}]")
        current = category
    print(f"  + {item}")

print("\n" + "=" * 62)
print("KEY FORMULAS")
print("=" * 62)
print("UEBA score       = 0.35*(new_resource) + 0.25*(off_hours) + 0.20*(vol_spike)")
print("                 + 0.10*(peer_dev) + 0.10*(velocity)")
print("Graph edge score = base(0.50) + velocity(new/3) + off_hours(0.20) + privilege(0.15)")
print("Session risk     = 0.20*(auth) + 0.25*(device) + 0.20*(network)")
print("                 + 0.25*(behavioral) + 0.10*(threat_intel)")
print("Entity match     = 0.35*(email) + 0.25*(name) + 0.20*(device)")
print("                 + 0.10*(dept) + 0.10*(phone)")
print("\nPDP decisions: < 0.25=ALLOW | 0.25-0.55=STEP-UP MFA | > 0.55=BLOCK+ALERT")
print("\nChapter 88 complete. Zero Trust with AI: not 'valid credential?' but 'right pattern?'")